# Collection A Processing: Split by Method (TMT, iTRAQ, Label-Free)

This notebook processes **Collection A** cohorts by grouping them into 3 distinct categories based on experimental method:
1.  **TMT** (CPTAC/ICPC/CNHPP)
2.  **iTRAQ** (Historical CPTAC)
3.  **Label-Free** (Other matched cohorts)

For each group, we:
- Load the cohort datasets.
- Concatenate them into a unified matrix.
- Filter Features (>50% missing) and Samples (>50% missing).
- Save separate integrated files for Protein and RNA.

In [1]:
import os
import glob
import pandas as pd
from tqdm import tqdm
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

DATA_DIR = r"./data/Collection_A"
OUTPUT_DIR = "./processed_data"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# --- DEFINING COHORT GROUPS ---
COHORT_GROUPS = {
    "TMT": [
        # CPTAC / ICPC / CNHPP
        "colon_2019", "renal_2019", "Endomet_val", "breast2020", "pedbrain_2020",
        "lung_2020_smoke", "lung_val", "lung2021", "headneck_2021", "pancrea2021",
        "brain_2021", "liver_val"
    ],
    "iTRAQ": [
        # Historical CPTAC
        "breast_2016", "ovary_2016", "stomach_2019"
    ],
    "LabelFree": [
        # Other matched cohorts
        "brain", "colon", "prostate", "liver", "lung"
    ]
}

# Initialize storage
group_data = {"TMT": {"prot": [], "rna": []}, "iTRAQ": {"prot": [], "rna": []}, "LabelFree": {"prot": [], "rna": []}}


## 1. Load Data into Groups

In [2]:
subdirs = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]
print(f"Found {len(subdirs)} subdirectories in {DATA_DIR}")

for subdir in tqdm(subdirs, desc="Processing Cohorts"):
    cohort_name = os.path.basename(subdir)
    
    # Determine Group
    assigned_group = None
    for group_name, cohort_list in COHORT_GROUPS.items():
        if cohort_name in cohort_list:
            assigned_group = group_name
            break
    
    if not assigned_group:
        print(f"[WARNING] Cohort '{cohort_name}' not in defined groups. Skipping.")
        continue

    # Find Files
    prot_files = glob.glob(os.path.join(subdir, "protein_matrix_*.txt"))
    rna_files = glob.glob(os.path.join(subdir, "RNA_matrix_*.txt"))
    
    if prot_files and rna_files:
        try:
            df_p = pd.read_csv(prot_files[0], sep="\t", index_col=0)
            df_r = pd.read_csv(rna_files[0], sep="\t", index_col=0)
            
            # Align Samples
            common_samples = df_p.columns.intersection(df_r.columns)
            
            if len(common_samples) > 0:
                df_p = df_p[common_samples]
                df_r = df_r[common_samples]

                # Rename Samples
                new_names = [f"{cohort_name.upper()}_{i+1}" for i in range(len(common_samples))]
                df_p.columns = new_names
                df_r.columns = new_names
                
                # Store
                group_data[assigned_group]["prot"].append(df_p)
                group_data[assigned_group]["rna"].append(df_r)
        except Exception as e:
            print(f"  -> ERROR in {cohort_name}: {e}")

Found 20 subdirectories in ./data/Collection_A


Processing Cohorts: 1


## 2. Concatenate, Filter, and Save per Group

In [3]:
def filter_nans(df, threshold=0.5):
    if df.empty: return df
    # 1. Rows (Features)
    df_clean = df.dropna(thresh=int(df.shape[1] * threshold), axis=0)
    rows_dropped = df.shape[0] - df_clean.shape[0]
    # 2. Cols (Samples)
    df_clean = df_clean.dropna(thresh=int(df_clean.shape[0] * threshold), axis=1)
    cols_dropped = df.shape[1] - df_clean.shape[1]
    print(f"   -> Dropped {rows_dropped} Features & {cols_dropped} Samples")
    return df_clean

for group_name, dfs in group_data.items():
    print(f"\n=== Processing Group: {group_name} ===")
    
    # Concatenate
    full_protein = pd.concat(dfs["prot"], axis=1, join='outer', sort=True) if dfs["prot"] else pd.DataFrame()
    full_rna = pd.concat(dfs["rna"], axis=1, join='outer', sort=True) if dfs["rna"] else pd.DataFrame()
    
    if full_protein.empty: 
        print(" [WARNING] No Protein data found.")
        continue
        
    print(f" Initial Shapes -> Protein: {full_protein.shape}, RNA: {full_rna.shape}")

    # Filter
    print(" Filtering Protein...")
    full_protein = filter_nans(full_protein)
    print(" Filtering RNA...")
    full_rna = filter_nans(full_rna)
    
    # Transpose (Samples x Features)
    integrated_protein = full_protein.T
    integrated_rna = full_rna.T
    
    # Remove duplicates
    integrated_protein = integrated_protein.loc[~integrated_protein.index.duplicated(keep='first')]
    integrated_rna = integrated_rna.loc[~integrated_rna.index.duplicated(keep='first')]
    
    print(f" Final Shapes   -> Protein: {integrated_protein.shape}, RNA: {integrated_rna.shape}")

    # Save
    p_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_Protein.csv")
    r_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_RNA.csv")
    
    integrated_protein.to_csv(p_out)
    integrated_rna.to_csv(r_out)
    print(f" Saved to {p_out} and {r_out}")


=== Processing Group: TMT ===
 Initial Shapes -> Protein: (11259, 1924), RNA: (33554, 1924)
 Filtering Protein...
   -> Dropped 4374 Features & 0 Samples
 Filtering RNA...
   -> Dropped 18553 Features & 0 Samples
 Final Shapes   -> Protein: (1924, 6885), RNA: (1924, 15001)
 Saved to ./processed_data\Collection_A_TMT_Protein.csv and ./processed_data\Collection_A_TMT_RNA.csv

=== Processing Group: iTRAQ ===
 Initial Shapes -> Protein: (7569, 276), RNA: (18859, 276)
 Filtering Protein...
   -> Dropped 2575 Features & 0 Samples
 Filtering RNA...
   -> Dropped 4337 Features & 0 Samples
 Final Shapes   -> Protein: (276, 4994), RNA: (276, 14522)
 Saved to ./processed_data\Collection_A_iTRAQ_Protein.csv and ./processed_data\Collection_A_iTRAQ_RNA.csv

=== Processing Group: LabelFree ===
 Initial Shapes -> Protein: (5767, 360), RNA: (25229, 360)
 Filtering Protein...
   -> Dropped 3204 Features & 0 Samples
 Filtering RNA...
   -> Dropped 10283 Features & 0 Samples
 Final Shapes   -> Protein: (